# Clase 222 — Surprise + Implicit + LightFM: benchmark comparativo

Mismo dataset sintético, las 3 librerías. NDCG@10 + tiempo de entrenamiento.

Requiere: `pip install scikit-surprise implicit lightfm scipy`.

In [ ]:
import os; os.environ['OPENBLAS_NUM_THREADS'] = '1'   # evitar warning de implicit
import numpy as np, pandas as pd, time
from scipy.sparse import csr_matrix

rng = np.random.default_rng(42)
n_users, n_items = 500, 300

# Generar ratings sintéticos
P_true = rng.normal(0, 1, (n_users, 8))
Q_true = rng.normal(0, 1, (n_items, 8))
scores_true = P_true @ Q_true.T
noise = rng.normal(0, 0.3, scores_true.shape)
interact_prob = 1 / (1 + np.exp(-(scores_true + noise)))
R = (rng.random(scores_true.shape) < 0.15 * interact_prob).astype(float)
ratings = np.where(R > 0, rng.integers(3, 6, R.shape), 0).astype(int)

df = pd.DataFrame([
    {'user_id': u, 'item_id': i, 'rating': int(ratings[u, i])}
    for u in range(n_users) for i in range(n_items) if ratings[u, i] > 0
])
print(f'dataset: {len(df):,} ratings')

## 1. Surprise SVD (explicit feedback)

In [ ]:
try:
    from surprise import Dataset, Reader, SVD
    from surprise.model_selection import train_test_split as surprise_split
    from surprise.accuracy import rmse

    reader = Reader(rating_scale=(1, 5))
    sd = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)
    trainset, testset = surprise_split(sd, test_size=0.2, random_state=42)

    t0 = time.perf_counter()
    algo = SVD(n_factors=50, n_epochs=20, random_state=42)
    algo.fit(trainset)
    train_t = time.perf_counter() - t0

    preds = algo.test(testset)
    surprise_rmse = rmse(preds, verbose=False)
    print(f'Surprise SVD: RMSE={surprise_rmse:.4f}, train={train_t:.2f}s')
except ImportError:
    print('pip install scikit-surprise')

## 2. Implicit ALS

In [ ]:
# Split temporal-ish (random aquí)
n = len(df)
idx = rng.permutation(n)
test_idx = set(idx[:n // 5].tolist())
train_mask = np.array([i not in test_idx for i in range(n)])
df_train, df_test = df.iloc[train_mask], df.iloc[~train_mask]

R_train = csr_matrix(
    (df_train.rating, (df_train.user_id, df_train.item_id)),
    shape=(n_users, n_items),
)
R_test = csr_matrix(
    (df_test.rating, (df_test.user_id, df_test.item_id)),
    shape=(n_users, n_items),
)

try:
    import implicit
    t0 = time.perf_counter()
    model_als = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, alpha=40, random_state=42)
    model_als.fit(R_train, show_progress=False)
    als_t = time.perf_counter() - t0
    print(f'Implicit ALS train: {als_t:.2f}s')

    t0 = time.perf_counter()
    model_bpr = implicit.bpr.BayesianPersonalizedRanking(factors=64, learning_rate=0.05, iterations=50, random_state=42)
    model_bpr.fit(R_train, show_progress=False)
    bpr_t = time.perf_counter() - t0
    print(f'Implicit BPR train: {bpr_t:.2f}s')
except ImportError:
    print('pip install implicit')

## 3. LightFM hybrid

In [ ]:
try:
    from lightfm import LightFM
    from lightfm.evaluation import precision_at_k as lfm_pak

    # Features sintéticos (géneros) — usar identity por default + 5 features extras
    item_feats = csr_matrix(np.hstack([
        np.eye(n_items),
        rng.binomial(1, 0.3, (n_items, 5)).astype(float),
    ]))

    t0 = time.perf_counter()
    model_lfm = LightFM(loss='warp', no_components=32, random_state=42)
    model_lfm.fit(R_train, item_features=item_feats, epochs=20, num_threads=2)
    lfm_t = time.perf_counter() - t0
    print(f'LightFM WARP train: {lfm_t:.2f}s')
except ImportError:
    print('pip install lightfm')

## 4. Evaluación uniforme: NDCG@10

In [ ]:
def ndcg_at_k(rel, k):
    rel_k = rel[:k]
    dcg = (rel_k / np.log2(np.arange(2, k + 2))).sum()
    ideal = np.sort(rel)[::-1][:k]
    idcg = (ideal / np.log2(np.arange(2, k + 2))).sum()
    return dcg / idcg if idcg > 0 else 0

def eval_implicit(model, R_train, R_test, k=10):
    R_test_dense = R_test.toarray()
    ndcgs = []
    for u in range(n_users):
        if R_test_dense[u].sum() == 0: continue
        ids, _ = model.recommend(u, R_train[u], N=k, filter_already_liked_items=True)
        rel = (R_test_dense[u, ids] > 0).astype(float)
        ndcgs.append(ndcg_at_k(rel, k))
    return float(np.mean(ndcgs))

results = []
try:
    results.append({'model': 'Implicit ALS', 'NDCG@10': eval_implicit(model_als, R_train, R_test), 'train_s': als_t})
    results.append({'model': 'Implicit BPR', 'NDCG@10': eval_implicit(model_bpr, R_train, R_test), 'train_s': bpr_t})
except NameError: pass

try:
    p_lfm = lfm_pak(model_lfm, R_test, train_interactions=R_train,
                    item_features=item_feats, k=10, num_threads=2).mean()
    results.append({'model': 'LightFM WARP', 'NDCG@10': float(p_lfm), 'train_s': lfm_t,
                    'note': '(precision@10 — proxy)'})
except NameError: pass

print(pd.DataFrame(results).round(4).to_string(index=False))

## 5. Decision matrix

In [ ]:
decision = pd.DataFrame([
    {'caso': 'aprender ABC / didáctico',           'recomendación': 'Surprise'},
    {'caso': 'producción CF, 10M-1B interactions', 'recomendación': 'Implicit ALS/BPR'},
    {'caso': 'hybrid con metadata rica',           'recomendación': 'LightFM'},
    {'caso': 'deep RS, escala TB, features ricos', 'recomendación': 'TF Recommenders / Spotlight'},
    {'caso': 'escala distribuida en Spark',        'recomendación': 'pyspark.ml.recommendation.ALS'},
    {'caso': 'serving embeddings <10ms p99',       'recomendación': 'FAISS (in-process) / Milvus (server)'},
])
print(decision.to_string(index=False))

## Ejercicio guiado

1. Replicá sobre MovieLens 1M real. Compará Surprise SVD vs Implicit ALS vs LightFM en NDCG@10 + tiempo.
2. Servir el mejor modelo: FastAPI (Clase 199) + FAISS index sobre `item_factors`. Medir p99 latency.
3. Probar TF Recommenders con tutorial oficial MovieLens — comparar effort vs Implicit.
4. Bonus: levantar `Milvus` en docker-compose. Indexar embeddings allí. Comparar con FAISS local.
5. Cost analysis: cuánto cuesta entrenar + servir 100M interactions con cada librería (cloud bill).

## Conclusiones

- 2026 default Python: **Implicit** para CF, **LightFM** si tenés features.
- TF Recommenders / two-tower para deep RS serio.
- Surprise solo educativo.
- Serving: FAISS para in-process, Milvus/Pinecone para gestionado.
- **Fin de Parte 6**: CF + content + hybrid + métricas + cold-start + librerías = stack completo de RS.